# GeoGami Street Morphology — Simplified Professor Workflow

This is the short practical workflow for editing, analysing, and comparing Env38 and Env39.

For the complete technical workflow, troubleshooting, QGIS details, Git safety, provenance, and scientific explanations, see:

`docs/professor_workflow.md`


# 1. Overall Workflow

```text
QGIS
  ↓
Edit Env38, Env39, or both
  ↓
Save edits
Turn editing OFF
Close QGIS completely
  ↓
Git commit the edited GeoPackage(s)
Repository must be clean
  ↓
Run analysis ONLY for the environment(s) that changed
  ↓
Each successful analysis creates:
- a NEW canonical run
- a NEW analysis run
- updated latest.json pointers
  ↓
Run:
python scripts/build_environment_comparison.py
  ↓
The comparison automatically uses:
latest accepted Env38
        VS
latest accepted Env39
  ↓
A NEW comparison report is created under:
results/comparison/env38_vs_env39/
```


# 2. Which Files Can Be Edited?

## Env38 — Curvilinear

Edit:

```text
data/editable/curvilinear/env38_editable.gpkg
```

## Env39 — Grid-like

Edit:

```text
data/editable/grid/env39_editable.gpkg
```

**Only edit the GeoPackages inside `data/editable/...`.**

Do not directly edit:

```text
data/canonical/
data/working/
results/analysis/
results/comparison/
```


# 3. Which Environment Do I Need to Analyse?

| What did I edit? | Run Env38 analysis? | Run Env39 analysis? |
|---|---:|---:|
| Only Env38 | Yes | No |
| Only Env39 | No | Yes |
| Both | Yes | Yes |
| Neither | No | No |

The comparison always uses the latest accepted result available for each environment.


# 4. Before Editing

Activate the Conda environment:

```bash
conda activate geogami-morphology
```

Update the repository:

```bash
git switch main
git pull
git status
```

The repository should be clean.

Create a branch before editing.

Example for Env38:

```bash
git switch -c edit-env38-YYYY-MM-DD
```

Example for Env39:

```bash
git switch -c edit-env39-YYYY-MM-DD
```

Example if both will be edited:

```bash
git switch -c edit-env38-env39-YYYY-MM-DD
```


# 5. Open the QGIS Project

Open:

```text
qgis/geogami_baselines.qgz
```

If the editable layers are not already loaded:

```text
Layer
→ Add Layer
→ Add Vector Layer
```

For Env38 choose:

```text
data/editable/curvilinear/env38_editable.gpkg
```

For Env39 choose:

```text
data/editable/grid/env39_editable.gpkg
```

Load:

```text
nodes
edges
```

Before editing, verify the layer source:

```text
Right-click layer
→ Properties
→ Information / Source
```

The source path should contain:

```text
data/editable/
```

Do not accidentally edit a layer from:

```text
data/canonical/
data/working/
```


# 6. What Can Be Changed in QGIS?

The professor may change the **geometry**.

Allowed:

```text
move existing nodes
reshape existing streets
change curvature
change orientation
change street length
add internal LineString vertices
move connected edge endpoints with a moved node
```

Do not change the graph topology.

Do not:

```text
add graph nodes
delete graph nodes
add streets
delete streets
split graph streets
merge graph streets
change node_id
change edge_id
change u
change v
change key
reconnect streets to different graph nodes
```

The scientific experiment is:

```text
same topology
different geometry
```


# 7. Curved Streets

A curved street does **not** require a new graph node.

Use internal LineString vertices to bend the street while keeping the same graph endpoints.

Example:

```text
N001 ---------------- N002
```

can become:

```text
N001 ---- curved geometry ---- N002
```

while remaining one physical graph edge.

A **graph node** represents network connectivity.

An **internal LineString vertex** only represents the shape of the street.


# 8. Trees, Playground, Buildings, etc.

Trees, playground graphics, buildings, water, landscape objects, and other Unity objects are not part of the NetworkX/OSMnx street graph.

Moving only a tree does not change:

```text
network topology
street length
circuity
orientation entropy
phi
NetworkX metrics
OSMnx metrics
```

If a visual object requires the street itself to move, then edit the **street geometry** in the editable GeoPackage.


# 9. Save and Close QGIS

Before running analysis:

```text
1. Save the layer edits.
2. Turn editing OFF.
3. Save the QGIS project if necessary.
4. Close QGIS completely.
```

Do not run the analysis while QGIS still has the GeoPackage open.

If files such as these exist:

```text
*.gpkg-wal
*.gpkg-shm
*.gpkg-journal
```

make sure QGIS is closed.

Do not manually delete active transaction files.


# 10. Commit the Edited GeoPackage

The production analysis requires a clean committed repository.

Check:

```bash
git status --short
```

## If Env38 was edited

Stage:

```bash
git add data/editable/curvilinear/env38_editable.gpkg
```

Commit:

```bash
git commit -m "Update Env38 street geometry"
```

## If Env39 was edited

Stage:

```bash
git add data/editable/grid/env39_editable.gpkg
```

Commit:

```bash
git commit -m "Update Env39 street geometry"
```

## If both were edited

Stage both:

```bash
git add data/editable/curvilinear/env38_editable.gpkg
git add data/editable/grid/env39_editable.gpkg
```

Then commit.

Finally check:

```bash
git status --porcelain
```

This should return nothing.

If it is not empty, review the changes before running production analysis.


# 11. If Only Env38 Was Changed

First run the Env38 dry-run:

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology --dry-run
```

If it ends with:

```text
FINAL RESULT: PASS
```

run the real Env38 analysis:

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology
```

This creates a new canonical Env38 run:

```text
data/canonical/curvilinear/runs/<NEW_ENV38_RUN>/
```

and a new Env38 analysis run:

```text
results/analysis/env38/<NEW_ENV38_RUN>/
```

It also updates:

```text
data/canonical/curvilinear/latest.json
results/analysis/env38/latest.json
```

Do **not** rerun Env39 if Env39 was not changed.

Then run:

```bash
python scripts/build_environment_comparison.py
```

The comparison automatically uses:

```text
NEW latest Env38
        VS
existing latest Env39
```


# 12. If Only Env39 Was Changed

First run the Env39 dry-run:

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology --dry-run
```

If it passes:

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology
```

This creates:

```text
data/canonical/grid/runs/<NEW_ENV39_RUN>/
```

and:

```text
results/analysis/env39/<NEW_ENV39_RUN>/
```

It updates:

```text
data/canonical/grid/latest.json
results/analysis/env39/latest.json
```

Do **not** rerun Env38 if Env38 was not changed.

Then run:

```bash
python scripts/build_environment_comparison.py
```

The comparison automatically uses:

```text
existing latest Env38
        VS
NEW latest Env39
```


# 13. If Both Env38 and Env39 Were Changed

Analyse Env38:

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology --dry-run
```

If PASS:

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology
```

Then analyse Env39:

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology --dry-run
```

If PASS:

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology
```

Then run:

```bash
python scripts/build_environment_comparison.py
```

It automatically compares:

```text
NEW latest Env38
        VS
NEW latest Env39
```


# 14. What Does `latest.json` Mean?

All historical runs remain saved.

`latest.json` is simply a pointer saying:

> This is the currently accepted/latest successful run for this environment.

## Env38

Canonical latest:

```text
data/canonical/curvilinear/latest.json
```

Analysis latest:

```text
results/analysis/env38/latest.json
```

## Env39

Canonical latest:

```text
data/canonical/grid/latest.json
```

Analysis latest:

```text
results/analysis/env39/latest.json
```

Therefore:

```bash
python scripts/build_environment_comparison.py
```

automatically knows which Env38 and Env39 runs to compare.

The professor does **not** normally need to enter run IDs manually.

The script does **not** compare every historical folder.


# 15. Where Are the Individual Results?

## Env38

Canonical runs:

```text
data/canonical/curvilinear/runs/<run_id>/
```

Analysis runs:

```text
results/analysis/env38/<run_id>/
```

## Env39

Canonical runs:

```text
data/canonical/grid/runs/<run_id>/
```

Analysis runs:

```text
results/analysis/env39/<run_id>/
```

Inside the analysis run there are folders/files for:

```text
graphs
topology
geometry
integrated
figures
CSV results
JSON results
manifests
```

Every successful analysis creates a **new run directory**.

Old runs are preserved.


# 16. Run the Env38 vs Env39 Comparison

After every environment that was changed has been successfully analysed:

```bash
python scripts/build_environment_comparison.py
```

The script automatically compares:

```text
latest accepted Env38
        VS
latest accepted Env39
```

A new comparison directory is created:

```text
results/comparison/env38_vs_env39/<comparison_id>/
```

Typical contents:

```text
comparison_metrics.csv
comparison_core_metrics.csv
comparison_summary.json
comparison_methodology.json
comparison_manifest.json
env38_vs_env39_summary.md
figures/
```

Old comparison folders are preserved.


# 17. Important Env38 Reproducibility Rule

The current accepted Env38 baseline has already passed independent reproducibility validation.

If Env38 geometry is changed and a completely new scientific Env38 baseline is created, the formal comparison may require a new reproducibility acceptance before that new Env38 run is allowed into the comparison.

The rigorous process is:

```text
Edit Env38
   ↓
Run Env38 analysis
   ↓
If required:
perform the Env38 reproducibility acceptance
   ↓
Env38 becomes comparison-eligible
   ↓
Run build_environment_comparison.py
```

If the comparison refuses a new Env38 run because reproducibility acceptance is missing, do not bypass the safeguard.

See:

```text
docs/professor_workflow.md
```

for the complete procedure.


# 18. Where Should I Look First?

## For Env38

Open the latest analysis run under:

```text
results/analysis/env38/<run_id>/
```

The most useful folder is:

```text
integrated/
```

## For Env39

Open:

```text
results/analysis/env39/<run_id>/
```

and look inside:

```text
integrated/
```

## For the Env38 vs Env39 Comparison

Open:

```text
results/comparison/env38_vs_env39/<comparison_id>/
```

Recommended files:

```text
env38_vs_env39_summary.md
comparison_core_metrics.csv
comparison_metrics.csv
figures/
```


# 19. Jupyter Notebooks

The command-line workflow already calculates and saves all scientific results.

The notebooks are mainly for:

```text
learning
inspection
visualization
interpretation
```

They are **not required** to produce the analysis.

Launch Jupyter:

```bash
conda activate geogami-morphology
python -m jupyter lab
```

Use the kernel:

```text
GeoGami Morphology
```

## Common Introduction

```text
00_osmnx_networkx_introduction.ipynb
```

## Env39

```text
01_env39_load_canonical_graph.ipynb
02_env39_topological_metrics.ipynb
03_env39_geometry_orientation_metrics.ipynb
04_env39_integrated_results.ipynb
```

## Env38

```text
05_env38_load_canonical_graph.ipynb
06_env38_topological_metrics.ipynb
07_env38_geometry_orientation_metrics.ipynb
08_env38_integrated_results.ipynb
```

The notebooks automatically use the current accepted `latest` run.


# 20. Commit the Generated Results

After a successful analysis:

```bash
git status --short
```

Review the new generated files.

## Example: Env38

Stage the new canonical and analysis outputs:

```bash
git add data/canonical/curvilinear/
git add results/analysis/env38/
```

Commit:

```bash
git commit -m "Record updated Env38 morphology analysis"
```

## Example: Env39

```bash
git add data/canonical/grid/runs/
git add data/canonical/grid/latest.json
git add results/analysis/env39/
```

Commit:

```bash
git commit -m "Record updated Env39 morphology analysis"
```

## If a New Comparison Was Created

```bash
git add results/comparison/env38_vs_env39/
```

Commit:

```bash
git commit -m "Record updated Env38 versus Env39 comparison"
```

Do not blindly stage unrelated files.


# 21. Push the Work to GitHub

Push the branch:

```bash
git push -u origin <branch-name>
```

Then:

```text
Create a Pull Request on GitHub
↓
Review the changes
↓
Run/check tests
↓
Merge after review
```


# 22. Quick Troubleshooting

## Repository is dirty

If the workflow says:

```text
Workflow requires a clean committed repository
```

run:

```bash
git status
```

Review and commit the intentional changes before production analysis.

---

## WinError 32 / File Is Being Used

Close QGIS completely.

Make sure no other application is using the GeoPackage.

Do not manually delete unknown SQLite transaction files.

---

## Topology-Control Failure

Check whether a graph element was accidentally:

```text
added
deleted
split
merged
reconnected
```

The standard workflow requires the same controlled topology.

---

## Endpoint Mismatch

An edge endpoint does not exactly coincide with its prescribed node.

Inspect the node and connected edge in QGIS and make them exactly coincident.

---

## Comparison Refuses a New Env38 Run

The new Env38 run may not yet have the required reproducibility acceptance.

Do not bypass the safeguard.

See:

```text
docs/professor_workflow.md
```


# 23. Three Simple Examples

## Example A — Only Env38 Changed

```text
Edit Env38 in QGIS
        ↓
Save + close QGIS
        ↓
Commit env38_editable.gpkg
        ↓
Env38 dry-run
        ↓
Env38 production analysis
        ↓
Do NOT rerun Env39
        ↓
Run comparison
        ↓
NEW Env38 latest
        VS
existing Env39 latest
```

## Example B — Only Env39 Changed

```text
Edit Env39 in QGIS
        ↓
Save + close QGIS
        ↓
Commit env39_editable.gpkg
        ↓
Env39 dry-run
        ↓
Env39 production analysis
        ↓
Do NOT rerun Env38
        ↓
Run comparison
        ↓
existing Env38 latest
        VS
NEW Env39 latest
```

## Example C — Both Changed

```text
Edit Env38 + Env39
        ↓
Save + close QGIS
        ↓
Commit both editable GeoPackages
        ↓
Run Env38 analysis
        ↓
Run Env39 analysis
        ↓
Run comparison
        ↓
NEW Env38 latest
        VS
NEW Env39 latest
```


# 24. Quick Command Cheat Sheet

## Activate Environment

```bash
conda activate geogami-morphology
```

## Git Status

```bash
git status --short
```

## Env38 Dry-Run

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology --dry-run
```

## Env38 Production

```bash
python scripts/run_full_analysis.py --environment env38 --input data/editable/curvilinear/env38_editable.gpkg --mode preserve-topology
```

## Env39 Dry-Run

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology --dry-run
```

## Env39 Production

```bash
python scripts/run_full_analysis.py --environment env39 --input data/editable/grid/env39_editable.gpkg --mode preserve-topology
```

## Env38 vs Env39 Comparison

```bash
python scripts/build_environment_comparison.py
```

## JupyterLab

```bash
python -m jupyter lab
```

## Complete Tests

```bash
conda run -n geogami-morphology python -m pytest
```

## Frozen Baseline Check

```bash
python scripts/verify_frozen_baselines.py
```


# 25. Remember the Main Idea

```text
QGIS changes the street geometry
        ↓
run_full_analysis.py analyses each changed environment
        ↓
latest.json identifies the current accepted results
        ↓
build_environment_comparison.py automatically compares:
latest Env38
VS
latest Env39
        ↓
A new versioned comparison report is created
```

Previous environment runs and previous comparison runs are preserved as historical research records.

The normal workflow never overwrites them.

For complete technical details, use:

```text
docs/professor_workflow.md
```
